# Ejercicio N°2

El dataset `ventas.xlsx` contiene los registros de una serie de ventas realizadas en el último tiempo en un local de productos electrónicos. Por otra parte, cuenta con el dataset `clientes_base.xlsx`, el cual contiene información sobre los clientes registrados en dicho establecimiento.

1. ¿Cuál fue el monto total de venta de productos iPad y MacBook?

2. Realice la unión de ambos DataFrames utilizando la operación que considere más adecuada y la columna `nombre_cliente` como key. ¿Qué observa en el DataFrame resultante?

3. Considerando que en `clientes_base.xlsx` los nombres de los clientes se encuentran exentos de errores ortográficos y tipográficos, ¿en qué porcentaje de los registros que conforman el dataset `ventas.xlsx` el nombre del cliente coincide con el de un cliente registrado?

4. Teniendo en cuenta lo observado en los ítems anteriores, utilice herramientas de fuzzy joins para realizar la unión de ambos datasets. ¿De qué ciudad es el cliente que más compras realizó en el local?

---



## Cargamos los data frames y analizamos los datos

In [1]:
# Librerías
import pandas as pd
from rapidfuzz import process, fuzz, utils

In [2]:
df_ventas = pd.read_excel('../../datasets/ventas.xlsx')
df_ventas.head(10)

,id_venta,nombre_cliente,producto,cantidad,precio_usd_producto
0,C1,Juana Perez,Apple Watch Series 8,2,399
1,C2,Roberto Gomezz,Nintendo Switch,1,299
2,C3,Carla Gonzáles Cuispe,Bose QuietComfort 45,1,329
3,C4,Jorge Martinez,Acer Predator Helios 300,1,1599
4,C5,Mariano Rodriguéz,iPad Pro,1,1099
5,C6,Roberto Gómez Acuña,HP Spectre x360,1,1399
6,C7,Maria Garcìa,MacBook Air,1,1249
7,C8,Carlos Gonzales,Samsung Galaxy S22,3,849
8,C9,Miguel Ánjel,Dell Alienware,1,1999
9,C10,María García,Amazon Echo Dot,3,49


In [3]:
df_ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id_venta             42 non-null     str  
 1   nombre_cliente       42 non-null     str  
 2   producto             42 non-null     str  
 3   cantidad             42 non-null     int64
 4   precio_usd_producto  42 non-null     int64
dtypes: int64(2), str(3)
memory usage: 1.8 KB


In [4]:
df_clientes = pd.read_excel('../../datasets/clientes_base.xlsx')
df_clientes.head()

,id_cliente,nombre_cliente,ciudad,email
0,1,Lucia Fernandez,Villa María,luciaf2@mail.com
1,2,Carlos Gómez,Mendoza,carlosgomez@mail.com
2,3,Andrés Pérez,Corrientes,andresp3@mail.com
3,4,Roberto Gómez,Rosario,rgomez@mail.com
4,5,Roberto Gómez Acuña,Corrientes,robgoac@mail.com


In [5]:
df_clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   id_cliente      36 non-null     int64
 1   nombre_cliente  36 non-null     str  
 2   ciudad          36 non-null     str  
 3   email           36 non-null     str  
dtypes: int64(1), str(3)
memory usage: 1.3 KB


## Ítem 1

¿Cuál fue el monto total de venta de productos iPad y MacBook?

In [6]:
df_ventas['producto'].unique()

<StringArray>
[     'Apple Watch Series 8',           'Nintendo Switch',
      'Bose QuietComfort 45',  'Acer Predator Helios 300',
                  'iPad Pro',           'HP Spectre x360',
               'MacBook Air',        'Samsung Galaxy S22',
            'Dell Alienware',           'Amazon Echo Dot',
            'Google Pixel 7',     'Microsoft Surface Pro',
               'AirPods Pro', 'Lenovo ThinkPad X1 Carbon',
        'Sony PlayStation 5',                 'iPad mini',
             'GoPro Hero 11',              'Canon EOS R5',
               'MacBook Pro',           'Google Nest Hub',
           'Sony WH-1000XM5',            'ASUS ROG Strix',
                  'iPad Air',            'Razer Blade 15',
               'Dell XPS 13',            'Bose SoundLink',
                'JBL Flip 5',    'Samsung Galaxy Watch 4',
            'OnePlus 10 Pro',           'Fitbit Charge 5',
      'Logitech MX Master 3',              'Xiaomi Mi 11',
            'Sony Bravia XR',         'Goo

In [7]:
# Filtramos las ventas que contengas iPad y MacBook en el nombre del producto:

ventas_filtrado = df_ventas[df_ventas['producto'].str.contains('iPad|MacBook')]
ventas_filtrado

,id_venta,nombre_cliente,producto,cantidad,precio_usd_producto
4,C5,Mariano Rodriguéz,iPad Pro,1,1099
6,C7,Maria Garcìa,MacBook Air,1,1249
15,C16,Laura Martínez,iPad mini,1,559
18,C19,Andres Pérrez,MacBook Pro,1,1999
22,C23,Andres Péres,iPad Air,2,599
38,C39,Juan Perez,iPad mini,1,559


In [8]:
# Calculamos el monto total de venta de estos productos:

monto_total = (ventas_filtrado['cantidad'] * ventas_filtrado['precio_usd_producto']).sum()

print(f'El monto total de venta de estos productos fue {monto_total} USD')

El monto total de venta de estos productos fue 6663 USD


## Ítem 2

Realice la unión de ambos DataFrames utilizando la operación que considere más adecuada y la columna `nombre_cliente` como key. ¿Qué observa en el DataFrame resultante?

In [9]:
# Podemos realizar la unión de los DataFrames utilizando la función merge de pandas, 
# pero como los nombres de los clientes no coinciden exactamente, podemos observar que hay 
# muchosvalores NaN resultado de la no coincidencia de los nombres.

df_union = pd.merge(df_ventas, df_clientes, on = 'nombre_cliente', how = 'left')
df_union.head(10)

,id_venta,nombre_cliente,producto,cantidad,precio_usd_producto,id_cliente,ciudad,email
0,C1,Juana Perez,Apple Watch Series 8,2,399,NaN,NaN,NaN
1,C2,Roberto Gomezz,Nintendo Switch,1,299,NaN,NaN,NaN
2,C3,Carla Gonzáles Cuispe,Bose QuietComfort 45,1,329,NaN,NaN,NaN
3,C4,Jorge Martinez,Acer Predator Helios 300,1,1599,NaN,NaN,NaN
4,C5,Mariano Rodriguéz,iPad Pro,1,1099,NaN,NaN,NaN
5,C6,Roberto Gómez Acuña,HP Spectre x360,1,1399,5.0,Corrientes,robgoac@mail.com
6,C7,Maria Garcìa,MacBook Air,1,1249,NaN,NaN,NaN
7,C8,Carlos Gonzales,Samsung Galaxy S22,3,849,NaN,NaN,NaN
8,C9,Miguel Ánjel,Dell Alienware,1,1999,NaN,NaN,NaN
9,C10,María García,Amazon Echo Dot,3,49,35.0,Córdoba,mariagarcia@mail.com


In [10]:
df_union.info()

<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id_venta             42 non-null     str    
 1   nombre_cliente       42 non-null     str    
 2   producto             42 non-null     str    
 3   cantidad             42 non-null     int64  
 4   precio_usd_producto  42 non-null     int64  
 5   id_cliente           14 non-null     float64
 6   ciudad               14 non-null     str    
 7   email                14 non-null     str    
dtypes: float64(1), int64(2), str(5)
memory usage: 2.8 KB


## Ítem 3

Considerando que en `clientes_base.xlsx` los nombres de los clientes se encuentran exentos de errores ortográficos y tipográficos, ¿en qué porcentaje de los registros que conforman el dataset `ventas.xlsx` el nombre del cliente coincide con el de un cliente registrado?

In [11]:
# Serie de Pandas con valores booleanos
coincidencias = df_ventas['nombre_cliente'].isin(df_clientes['nombre_cliente'].unique())

# Total de registros coincidentes en el nombre
total_coincidencias = coincidencias.sum()

# Calculamos porcentaje sobre el número de registros de ventas
porcentaje = round(total_coincidencias*100/len(df_ventas), 2)

print(f'En el {porcentaje} % de los registros de ventas coincide el nombre del cliente con el de un cliente registrado')

En el 33.33 % de los registros de ventas coincide el nombre del cliente con el de un cliente registrado


## Ítem 4

Teniendo en cuenta lo observado en los ítems anteriores, utilice herramientas de fuzzy joins para realizar la unión de ambos datasets. ¿De qué ciudad es el cliente que más compras realizó en el local?

In [12]:
def buscar_coincidencia(nombre, opciones, umbral = 80):
    """
    Esta función recibe un nombre de búsqueda (query), una lista de 
    candidatos (choices) y un scorer, y devuelve el candidato más 
    similar junto con su score. Si ningún candidato supera el umbral 
    mínimo indicado, devuelve None.
    """
    resultado = process.extractOne(
        nombre,
        opciones,
        scorer = fuzz.token_sort_ratio,
        processor = utils.default_process,
        score_cutoff = umbral
    )
    if resultado:
        return resultado[0], round(resultado[1],2) 
    return "Ningún match", "No aplica"


df_clientes[['nombre_matched', 'score']] = df_clientes['nombre_cliente'].apply(
    lambda x: pd.Series(buscar_coincidencia(x, df_ventas['nombre_cliente']))
)

# Aplicamos la función a cada fila de df_beneficiarios:

print(df_clientes[['nombre_cliente', 'nombre_matched', 'score']].head())

        nombre_cliente       nombre_matched  score
0      Lucia Fernandez      Lucía Fernàndez  86.67
1         Carlos Gómez         Carlos Gomez  91.67
2         Andrés Pérez        Andres Pérrez   88.0
3        Roberto Gómez        Roberto Gómez  100.0
4  Roberto Gómez Acuña  Roberto Gómez Acuña  100.0


In [13]:
# Unimos los DataFrames utilizando la columna de nombres coincidentes:

df_final = df_clientes.merge(df_ventas, left_on='nombre_matched', right_on='nombre_cliente') \
                                    .drop(columns=['nombre_matched', 'score']) \
                                    .rename(columns={'nombre_cliente_x': 'nombre'})

df_final.head(10)

,id_cliente,nombre,ciudad,email,id_venta,nombre_cliente_y,producto,cantidad,precio_usd_producto
0,1,Lucia Fernandez,Villa María,luciaf2@mail.com,C29,Lucía Fernàndez,OnePlus 10 Pro,1,799
1,2,Carlos Gómez,Mendoza,carlosgomez@mail.com,C40,Carlos Gomez,Sony Alpha A7 IV,1,2499
2,3,Andrés Pérez,Corrientes,andresp3@mail.com,C19,Andres Pérrez,MacBook Pro,1,1999
3,4,Roberto Gómez,Rosario,rgomez@mail.com,C15,Roberto Gómez,Sony PlayStation 5,1,499
4,5,Roberto Gómez Acuña,Corrientes,robgoac@mail.com,C6,Roberto Gómez Acuña,HP Spectre x360,1,1399
5,6,Juana Pérez,Salta,juanaperez@mail.com,C1,Juana Perez,Apple Watch Series 8,2,399
6,7,Lucía Hernández,Santa Fe,luciahernandez@mail.com,C21,Lucía Hernándezz,Sony WH-1000XM5,1,399
7,8,Andrés Pérez Gollán,Mar del Plata,andresp@mail.com,C11,Andrés Pérez Gollán,Google Pixel 7,2,599
8,9,Miguel Ángel,Neuquén,miguelangel@mail.com,C9,Miguel Ánjel,Dell Alienware,1,1999
9,10,Marcos Rupetti,Bariloche,marcosg@mail.com,C41,Marcos Rupetti,LG OLED TV,1,1499


In [14]:
# Respondemos a la pregunta por la ciudad del cliente que más compras realizó:

df_final['nombre'].value_counts().head()

nombre
Juan Pérez         2
Lucia Fernandez    1
Carlos Gómez       1
Andrés Pérez       1
Roberto Gómez      1
Name: count, dtype: int64